In [ ]:
# Weather Pattern Forecast Probability Plots

Reads ECMWF ensemble weather-pattern (WP) forecast CSV files and plots the
probability of high-risk patterns (HRPs) at each lead time, alongside the
monthly climatological probability.

**Inputs**
| File | Description |
|---|---|
| `data/forecasts/{Event}/Ensemble_Matrix15_{date}_00_ecmwf_30regimes.csv` | Per-run ECMWF WP ensemble matrix (15 lead days × 51 members) |
| `data/preliminary_classifications_2024_30regimes.csv` | Historical WP classifications used to compute climatology |

**Outputs**
| File | Description |
|---|---|
| `output/figures/WP_probability_grid.png` | 3×3 subplot grid (events × lead times) |
| `output/figures/WP_probability_{event}_{lead}.png` | Individual forecast plots (one per event/lead combination) |

**Dependencies:** `pattern_climatologies` — a local module providing
`three_monthly()` and `combined_pattern_climatology()` helpers. Ensure it is
on `sys.path` before running.

**Configuration:** Edit the paths and event dictionary below before running.


In [ ]:
import os
import sys
import string
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── User configuration ──────────────────────────────────────────────────────
# Directory containing per-event forecast CSV subdirectories
FORECAST_BASE_DIR = "data/forecasts"

# Historical WP classification CSV (used for climatology)
HIST_CLASSIFICATIONS = "data/preliminary_classifications_2024_30regimes.csv"

# Output directory for figures
FIG_DIR = "output/figures"
os.makedirs(FIG_DIR, exist_ok=True)

# High-risk weather patterns most associated with CD/CW occurrence
HRP_LIST = [19, 27, 28]
BAR_COLOURS = ('#44519F', '#008178', '#68CFE0')

# Forecast configuration
NLEADS    = 15    # number of forecast lead days
NMEMBERS  = 51    # number of ensemble members
NPATTERNS = 30    # total number of weather patterns
RUN_HOUR  = '00'  # forecast initialisation hour

# Events: each entry has three lead-time runs (10-day, 5-day, 3-day)
# date format matches the CSV filename: DDMMYYYY
EVENTS_DICT = {
    "Event_1_11Feb2021": [
        {"lead_time": "10_days", "date": "01022021"},
        {"lead_time": "5_days",  "date": "06022021"},
        {"lead_time": "3_days",  "date": "08022021"},
    ],
    "Event_2_28Feb2018": [
        {"lead_time": "10_days", "date": "18022018"},
        {"lead_time": "5_days",  "date": "23022018"},
        {"lead_time": "3_days",  "date": "25022018"},
    ],
    "Event_3_29Nov2010": [
        {"lead_time": "10_days", "date": "19112010"},
        {"lead_time": "5_days",  "date": "24112010"},
        {"lead_time": "3_days",  "date": "26112010"},
    ],
}

# Row labels (event dates) and column labels (lead times) for the grid figure
EVENT_ROW_DATES  = ['11/02/21', '28/02/18', '29/11/10']
COL_LEAD_TEXTS   = ['10 day lead time', '5 day lead time', '3 day lead time']

print("Configuration complete.")


## 1. Load climatology

In [ ]:
import pattern_climatologies   # local module — ensure it is on sys.path

# monthly_regime_climatologies shape: (nRegimes, nMonths)
monthly_regime_climatologies = pattern_climatologies.three_monthly(HIST_CLASSIFICATIONS)
print("Climatology loaded. Shape:", np.array(monthly_regime_climatologies).shape)


## 2. Helper: load and process one forecast CSV

In [ ]:
def load_forecast(event_name, date_str):
    """
    Load an ECMWF WP ensemble matrix CSV and return the assigned regimes,
    validity dates, and run date/time.

    Parameters
    ----------
    event_name : str   Key from EVENTS_DICT (used to locate the subdirectory)
    date_str   : str   Date string in DDMMYYYY format

    Returns
    -------
    assigned_regimes : ndarray (NLEADS, NMEMBERS)  Integer WP per member/lead
    day              : list of datetime             Validity dates (NLEADS entries)
    month_list       : list of int                  Month for each validity date
    run_date_time    : datetime                     Initialisation date/time
    """
    csv_path = os.path.join(FORECAST_BASE_DIR, event_name,
                            f"Ensemble_Matrix15_{date_str}_00_ecmwf_30regimes.csv")
    df = pd.read_csv(csv_path, header=None)
    df_values = df.iloc[:-1, 1:]
    assigned_regimes = df_values.to_numpy(dtype=int)

    if assigned_regimes.shape != (NLEADS, NMEMBERS):
        print(f"Warning: unexpected shape {assigned_regimes.shape} for {csv_path}")

    date_column   = df.iloc[:-1, 0]
    run_date_time = pd.to_datetime(date_column.iloc[0]).to_pydatetime()
    run_date_time = run_date_time.replace(hour=int(RUN_HOUR))

    initial_diff = datetime.timedelta(hours=12 if RUN_HOUR == '00' else 24)
    time_step    = datetime.timedelta(hours=24)
    day = []
    for ilead in range(NLEADS):
        if ilead == 0:
            day.append(run_date_time + initial_diff)
        else:
            day.append(day[-1] + time_step)

    month_list = [dt.month for dt in day]
    return assigned_regimes, day, month_list, run_date_time


def compute_hrp_probabilities(assigned_regimes, hrp_list):
    """
    Compute forecast probabilities (%) for each HRP at each lead day.

    Returns ndarray of shape (NLEADS, len(hrp_list)).
    """
    probs = np.zeros((NLEADS, len(hrp_list)), dtype=float)
    for i, pattern in enumerate(hrp_list):
        for ilead in range(NLEADS):
            probs[ilead, i] = np.sum(assigned_regimes[ilead] == pattern) / NMEMBERS * 100.0
    return probs


## 3. Figure 1 — 3×3 subplot grid (events × lead times)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()
plot_idx = 0

for event, entries in EVENTS_DICT.items():
    for entry in entries:
        if plot_idx >= len(axes):
            break

        assigned_regimes, day, month_list, run_date_time = load_forecast(
            event, entry['date']
        )
        probs = compute_hrp_probabilities(assigned_regimes, HRP_LIST)

        # Climatology for this forecast window
        monthly_probs = pattern_climatologies.combined_pattern_climatology(
            HRP_LIST, monthly_regime_climatologies)
        climatology = [monthly_probs[m - 1] for m in month_list]

        ax = axes[plot_idx]
        x      = np.arange(NLEADS) + 1
        bottom = np.zeros(NLEADS)

        for i, pattern in enumerate(HRP_LIST):
            h = probs[:, i]
            mask = h > 0
            if np.any(mask):
                ax.bar(x[mask], h[mask], width=0.8, bottom=bottom[mask],
                       color=BAR_COLOURS[i], edgecolor='k', linewidth=1)
                bottom[mask] += h[mask]

        ax.plot(x, climatology, linewidth=2, color='darkred',
                alpha=0.6, linestyle='dashed')

        ax.set_xlim(0.5, NLEADS + 0.5)
        ax.set_ylim(0, 100)
        ax.set_yticks(range(0, 110, 20))
        ax.set_xticks(x)
        ax.set_xticklabels(
            [day[i].strftime("%a
%d
%b") for i in range(NLEADS)], fontsize=8
        )
        ax.grid(axis='y', linestyle=':', color='grey')
        ax.set_ylabel('Probability (%)', fontsize=9)

        label    = f"({string.ascii_lowercase[plot_idx]})"
        date_str = run_date_time.strftime("%a %d %b %Y")
        ax.set_title(f"{label} {date_str}", loc='left', fontsize=10)

        plot_idx += 1

# Shared legend
hrp_handles = [mpatches.Rectangle((0, 0), 1, 1, color=BAR_COLOURS[i], ec='k')
               for i in range(len(HRP_LIST))]
clim_handle = plt.Line2D([0], [0], color='darkred', lw=2, linestyle='dashed')
legend_labels = [f'Pattern {p}' for p in HRP_LIST] + ['Climatology']

fig.legend(hrp_handles + [clim_handle], legend_labels,
           loc='upper center', bbox_to_anchor=(0.5, 1.03),
           ncol=len(legend_labels), frameon=True,
           fontsize='small',
           title='High-risk weather patterns relevant to CD/CW occurrence',
           title_fontsize='10')

plt.tight_layout(rect=[0, 0, 1, 0.96])

# Row event-date labels (rotated, left of each row)
for row, label in enumerate(EVENT_ROW_DATES):
    left_ax = axes[row * 3]
    pos = left_ax.get_position()
    fig.text(pos.x0 + 0.01, pos.y0 + pos.height / 2,
             label, rotation=90, ha='center', va='center',
             fontsize=12, fontweight='bold')

# Column lead-time labels above the top row
for col, text in enumerate(COL_LEAD_TEXTS):
    top_ax = axes[col]
    pos = top_ax.get_position()
    fig.text(pos.x0 + pos.width / 2, pos.y1 + 0.02,
             text, ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.subplots_adjust(left=0.12)
plt.savefig(os.path.join(FIG_DIR, 'WP_probability_grid.png'),
            dpi=300, bbox_inches='tight')
plt.show()


## 4. Figure 2 — Individual forecast plots (one per event/lead combination)

In [ ]:
for event, entries in EVENTS_DICT.items():
    for entry in entries:
        lead_time = entry['lead_time']
        assigned_regimes, day, month_list, run_date_time = load_forecast(
            event, entry['date']
        )

        # Full 30-pattern probabilities for annotating the most-probable WP
        full_probs = np.zeros((NLEADS, NPATTERNS), dtype=float)
        for ipattern in range(NPATTERNS):
            for ilead in range(NLEADS):
                full_probs[ilead, ipattern] = (
                    np.sum(assigned_regimes[ilead] == ipattern) / NMEMBERS * 100.0
                )

        # HRP-only probabilities for plotting
        probs = compute_hrp_probabilities(assigned_regimes, HRP_LIST)

        # Climatology
        monthly_probs = pattern_climatologies.combined_pattern_climatology(
            HRP_LIST, monthly_regime_climatologies)
        climatology = [monthly_probs[m - 1] for m in month_list]

        # Build figure
        fig = plt.figure(figsize=(8, 7), dpi=100)
        ax  = fig.add_axes([0.12, 0.20, 0.78, 0.487])

        x      = np.arange(NLEADS) + 1
        bottom = np.zeros(NLEADS)

        # Stacked bars — HRPs only
        p_stack = []
        for i, pattern in enumerate(HRP_LIST):
            h = probs[:, i]
            if i == 0:
                p_stack.append(ax.bar(x, h, 0.8, align='center',
                                      color=BAR_COLOURS[i],
                                      linewidth=1, edgecolor='k'))
            else:
                p_stack.append(ax.bar(x, h, 0.8, align='center',
                                      color=BAR_COLOURS[i], linewidth=1,
                                      edgecolor='k',
                                      bottom=np.sum(probs[:, :i], axis=1)))

        # Climatology line
        clim_line = ax.plot(x, climatology, linewidth=2, color='darkred',
                            label='Climatology', alpha=0.6, linestyle='dashed',
                            zorder=999)
        p_stack.append(clim_line[0])

        # Hatching for short-range lead times
        ax.axvspan(0.5, 2.5, fill=False, edgecolor='black',
                   alpha=0.2, linewidth=1.0, hatch='/')
        ax.text(0.52, -30.7,
                'Hatching indicates forecast lead times when
'
                'higher resolution forecasts should be used.',
                fontsize='10', alpha=0.5)

        legend_labels_ind = [f'Pattern {p}' for p in HRP_LIST] + ['Climatology']
        legend = ax.legend(p_stack, legend_labels_ind,
                           bbox_to_anchor=(0., 1.02, 1., .102),
                           loc=3, mode='expand', ncol=len(legend_labels_ind) + 1,
                           borderaxespad=0., labelspacing=0.4,
                           columnspacing=0.05, handletextpad=0.2,
                           edgecolor='black', fancybox=False, borderpad=0.4,
                           title='High-risk weather patterns most relevant to CD/CW occurrence')
        legend.get_title().set_fontsize('10')
        for lbl in legend.get_texts():
            lbl.set_fontsize('small')

        xlabels = [' '] + [
            day[i].strftime("%a") + '
' + str(day[i].day) + '
' + day[i].strftime("%b")
            for i in range(NLEADS)
        ]
        ax.set_xticks(range(len(xlabels)))
        ax.set_xticklabels(xlabels, fontsize=11, rotation=0)
        ax.set_xlim(0.5, NLEADS + 0.5)
        ax.set_ylim(0, 100)
        ax.set_yticks(range(0, 110, 10))
        ax.tick_params(axis='y', labelsize=11)
        ax.yaxis.grid(True, color='grey', linestyle=':')
        ax.xaxis.grid(False)
        ax.set_ylabel('Probability (%)', fontsize=11)

        run_str = (run_date_time.strftime("%a") + ' ' + str(run_date_time.day)
                   + ' ' + run_date_time.strftime("%b") + ' ' + str(run_date_time.year))
        ax.set_title(
            f'CD/CW Decider for Scotland
Probability of high-risk weather patterns
{run_str}




',
            x=0.5, fontsize='12'
        )

        safe_event = event.replace(' ', '_')
        fig.savefig(os.path.join(FIG_DIR, f"WP_probability_{safe_event}_{lead_time}.png"),
                    dpi=300, bbox_inches='tight')
        plt.show()
